# Real-data direct multi-horizon forecast benchmark

This local-only research notebook compares independent and coupled TsgamForecastEstimator fits on bundled PV solar, ISO load, and tidal water-level data. It performs no downloads.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

examples_dir = Path.cwd() / 'examples'
if not (examples_dir / 'forecast_real_data_support.py').exists():
    examples_dir = Path.cwd()
if str(examples_dir) not in sys.path:
    sys.path.insert(0, str(examples_dir))

from forecast_real_data_support import (
    compact_summary,
    load_all_datasets,
    metrics_table,
    overview_table,
    plot_forecast_paths,
    plot_horizon_metrics,
    plot_roughness,
    run_all_benchmarks,
    split_and_alignment_table,
)

sns.set_theme(style='whitegrid', context='notebook')

## Configuration

These are deterministic bounded windows at native source frequency. The final horizon rows are reserved only for held-out targets.

In [ ]:
DATASET_NAMES = ('pv_solar', 'iso_load', 'tidal_water_level')
FORECAST_MODES = ('independent', 'coupled')
FORECAST_PATH_ORIGIN_FRACTION = 0.5
LOCAL_DATA_ONLY = True

## Local data, known-at-origin information, and missing-data policy

Every feature is observed at the forecast origin. Targets are never filled; source features are forward-filled only up to a short dataset-specific limit, so no future feature value reaches an origin. The loaders preserve the modal source grid and do not resample.

In [ ]:
datasets = load_all_datasets(DATASET_NAMES)
display(overview_table(datasets))

## Train/test origins and target alignment

For origin o and horizon h, direct multi-horizon fitting uses X(o) to y(o + h). The table shows the split plus concrete examples of the origin, known-feature timestamp, and scored target timestamp, including the h=0 nowcast.

In [ ]:
display(split_and_alignment_table(datasets))

## Run the benchmark

Independent and coupled modes use the same origin-time basis. Coupling adds a first-difference penalty across positive-horizon coefficients; h=0 remains an uncoupled diagnostic baseline. Lower coefficient roughness is a smoothness result rather than a guarantee of lower error.

In [ ]:
benchmark_started = perf_counter()
results = run_all_benchmarks(datasets)
total_runtime_seconds = perf_counter() - benchmark_started
print(f'Completed {len(results)} bounded local benchmarks in {total_runtime_seconds:.1f} seconds.')

## Horizon-wise RMSE and MAE

Horizon zero is the aligned nowcast baseline; positive horizons are future forecasts from the same origin rows.

In [ ]:
metrics = metrics_table(results)
display(metrics)
metric_figure = plot_horizon_metrics(results)
plt.show()

## Forecast paths

Each panel shows observed history, the h=0 nowcast at the forecast origin, and the future target path. Prediction rows are indexed by origin and unwrapped onto their target timestamps for this view.

In [ ]:
path_figure = plot_forecast_paths(
    results,
    origin_fraction=FORECAST_PATH_ORIGIN_FRACTION,
)
plt.show()

## Coefficient roughness and compact summary

In [ ]:
roughness_figure = plot_roughness(results)
plt.show()

summary = compact_summary(results)
display(summary)

## Caveats

These are small reproducible research windows, not a production backtest. Runtime depends on the local CVXPY solver. The ISO workbook does not include a metered solar or net-load series, so its result is reported honestly as a real-time load benchmark.